<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex12.1-power-grid-stability-estimation/Ex12.1_05_compare_and_report.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_12.1 · Notebook 05 — Compare, and Write It Up

**Paired with L12.1 · Power Grid Stability Estimation**

**Prerequisite: all previous notebooks.**

This notebook collects what you produced and turns it into the report. It
computes almost nothing new — its job is to make you put the numbers next to
each other, which is where the conclusions actually are.

---

## 0 · Setup

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex12.1-power-grid-stability-estimation/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
# outputs-cell v1 --------------------------------------------------------
# Later notebooks in this set read results that earlier ones save. On Google
# Colab every notebook runs on its own temporary machine, so a file saved
# here is not there when the next notebook opens. This cell keeps the
# results in your Google Drive instead: approve the access request when it
# appears. If you decline it, or have no Google Drive, the results are
# downloaded to your computer when saved and the notebook that needs them
# asks for them back. Locally this cell does nothing.
import course_core as cc
import problem as pb
pb.RESULTS = cc.keep_outputs("Ex12.1_outputs")


In [ ]:
# --- setup: every Part 2 notebook opens with this cell ------------------
# Needs course_core.py, pinn_core.py and problem.py beside this notebook.
# On Colab the files cell above fetched them from the public course repository.
import os
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"

from pinn_core import *                                  # noqa: F401,F403
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt

set_seed(88)
print("device:", DEVICE, " dtype:", torch.get_default_dtype())

In [ ]:
ref = pb.load("00_reference")
wls = pb.load("01_wls")
alg = pb.load("02_algebraic")
dyn = pb.load("03_dynamic")
inr = pb.load("04_inertia")

V_true, th_true = ref["V"], ref["th"]
ms_thin = pb.thin_measurements()
metered = ms_thin.measured_buses()

r_wls = pb.error_table(V_true, th_true, wls["V_thin"], wls["th_thin"],
                       metered, label="WLS, thin set")
r_pinn = pb.error_table(V_true, th_true, alg["V_pinn"], alg["th_pinn"],
                        metered, label="PINN, thin set")
pb.comparison_table([("WLS (thin)", r_wls), ("PINN (thin)", r_pinn)])

**Expected output**

> The PINN's unmetered-bus error should be clearly below WLS's. If it is not,
> say so — an honest negative, investigated, is worth more than a tuned positive,
> and the first thing to check is your λ.

## The five relations that must hold

If any of these comes out backwards, something is wrong with the
implementation rather than interesting about the physics. This is the fastest
debugging tool in the exercise.

In [ ]:
checks = [
    ("full metering: WLS is accurate",            None),
    ("thin metering: WLS degrades badly",         None),
    ("thin metering: PINN beats WLS unmetered",   None),
    ("quiet window: inertia estimate unreliable", None),
    ("wrong topology: small residual, wrong state", None),
]
# TODO: replace each None with True/False computed from your own results,
#       and print the table. Do not hand-wave: each one is checkable.
for name, val in checks:
    print(f"  [{'?' if val is None else ('OK' if val else 'FAIL')}]  {name}")

## TODO — the report

Fill in the sections below and generate the document. The final question is the
one that carries the marks.

In [ ]:
sections = [
    ("Measurement sets and observability",
     "TODO: state both sets, their ranks, and which buses were unmetered."),
    ("WLS baseline",
     "TODO: your numbers from notebook 01, both measurement sets."),
    ("The lambda sweep",
     "TODO: the curve, the value you chose, and why. State what happens at "
     "both extremes."),
    ("Estimation error, split",
     "TODO: metered vs unmetered, worst and mean. Not the mean alone."),
    ("Inertia",
     "TODO: your H for machine 1, the window you used, and how much you trust "
     "it. Include the quiet-window result."),
    ("RoCoF and its window",
     "TODO: your three window values and what you concluded."),
    ("What depends on the assumed impedances",
     "TODO: the network is DK2-representative. Which of your conclusions "
     "would survive if the reactances were wrong by 20 per cent?"),
]
pb.make_report(sections,
               os.path.join(pb.RESULTS, "Ex12.1_report.md"),
               author="TODO: your name")

**Expected output**

> `wrote Ex12.1_outputs/Ex12.1_report.md`
>
> Open it, fill in every TODO, and read the last section twice before you submit.

---

## Before you hand it in

1. Every table separates metered from unmetered buses, and reports the worst
   bus as well as the mean. If one does not, that table is misleading and you
   know it.
2. Every number carries the seed and the measurement set it came from.
3. The λ you chose is justified in a sentence, not asserted.
4. The inertia you report is machine 1's, and the reason you are not reporting
   machine 0's is written down.
5. You have said which of your conclusions depend on the assumed line
   impedances, and which depend only on the structure of the network.

<!-- notebook-questions v1 -->
---

## The questions from notebooks 01 to 04

Every notebook in this set ended with four questions under *Before you move
on*. Copy your answers to them into the cell below — a few sentences each —
and the cell after it adds all 16, each under its question, to the end of the
report you just wrote. On Colab every notebook runs on its own machine, so this
notebook cannot read what you wrote in the others: copying is the only way
across.

Keep the answers short and in your own words. The arrow after each question
names the question on the lecture's Questions slide that it helps answer, so
this section is also your preparation for the oral examination.

In [ ]:
# notebook-questions v1 -- your answers from notebooks 01 to 04 ----------
# Paste each answer between its triple quotes. The question is in the
# comment above it; an empty answer is reported as not answered.

NOTEBOOK_ANSWERS = {

    # ---- notebook 01 · Weighted Least Squares, the Baseline ----------------------
    # 01.1 With the default set WLS converged with J around 9.5 and a worst |V|
    # error of a few thousandths. With the thin set it converged just as
    # readily and was roughly 45 times worse, worst where there were no meters.
    # WLS has run for fifty years: say what these two runs show it does well,
    # what it cannot do, and why its convergence report gave you no warning.
    # (-> L12.1 Q10)
    "01.1": """
""",
    # 01.2 The thin set gives rank 4 of the 10 unknowns. Say what observability
    # depends on and what it does not, and use this set to explain why a more
    # accurate meter at the same two buses would not have helped. What was the
    # ridge term deciding at the unmetered buses? (-> L12.1 Q3)
    "01.2": """
""",
    # 01.3 Section 1 could score every estimate only because it manufactured
    # the readings from a known truth. In TODO 1 there was still a way to find
    # the bad meter. Name the diagnostic, say how it checks an estimate without
    # any ground truth, and name one error it cannot catch however large it is.
    # (-> L12.1 Q6)
    "01.3": """
""",
    # 01.4 TODO 2 tried every one-meter addition to the thin set. Explain the
    # winning placement in terms of the graph (which buses it ties to a
    # measurement, which unobservable part it breaks) rather than the size of
    # that bus's load, and say why an exhaustive search is the right claim to
    # make on six buses and not on six hundred. (-> L12.1 Q3)
    "01.4": """
""",

    # ---- notebook 02 · Estimation With the Physics as a Residual -----------------
    # 02.1 At $\lambda_{\mathrm{pf}} = 0$ this estimator is least squares on
    # the meters; very large, it becomes a power flow solution that ignores
    # them. Use the two ends of your $\lambda$ sweep to explain why the power
    # flow equations enter here as a residual to be satisfied rather than as a
    # simulation you run, and why the trade-off only showed at the unmetered
    # buses. (-> L12.1 Q1)
    "02.1": """
""",
    # 02.2 On the same thin set WLS left the unmetered buses about 3.0e-2 out.
    # Explain how adding $\mathcal{L}_{\mathrm{pf}}$ gives the optimiser
    # information at a bus with no meter at all, and why the result there must
    # still be reported as an inference and not as a reading. (-> L12.1 Q4, Q3)
    "02.2": """
""",
    # 02.3 TODO 2 gave the estimator an admittance matrix with a branch
    # missing. Say what the admittance matrix contains, where it comes from,
    # and which part of this estimator, if any, is learned. Then use the
    # wrong-Y run to explain why the loss did not warn you, and what a control
    # room could check instead. (-> L12.1 Q2, Q6)
    "02.3": """
""",
    # 02.4 Two things are built in here: the slack angle, and a sigmoid bound
    # of 0.90 to 1.10 per unit on the voltage magnitudes. Say why the slack
    # angle can be built in exactly, why the lecture's 0.94 to 1.06 operating
    # band must only be checked, and what the wider bound would hide if a bus
    # really fell below 0.90. Then say what a graph network would add over this
    # direct solve, and why the notebook uses none. (-> L12.1 Q5, Q8)
    "02.4": """
""",

    # ---- notebook 03 · Letting the Grid Move -------------------------------------
    # 03.1 Notebooks 01 and 02 treated the grid as a snapshot. Using section
    # 1's trajectory and section 2's critical clearing time of about 0.33 s,
    # say what the swing equation adds over power flow: name a question an
    # operator needs answered that the power flow equations cannot answer at
    # all. (-> L12.1 Q9)
    "03.1": """
""",
    # 03.2 Frequency is measured at scattered instants and the load angle is
    # never measured anywhere. Explain how the swing residual turns the
    # frequency record into an angle trajectory, and why neither the data term
    # alone nor the residual alone would determine it. How is that the same
    # move as notebook 02's unmetered buses? (-> L12.1 Q4, Q9)
    "03.2": """
""",
    # 03.3 There are no labels for $\delta$, so TODO 2's residual between
    # samples is your check. Where was it largest, and what would you conclude
    # if it were large everywhere? Say how this and notebook 01's normalised
    # residuals together let you check an estimate with no ground truth. (->
    # L12.1 Q6)
    "03.3": """
""",
    # 03.4 Machine 0 stands for the Nordic system seen through the Swedish
    # link, with an inertia so large that it barely moves. Explain why eastern
    # Denmark's frequency is set by that system and not by western Denmark's,
    # and how an HVDC link would enter this model if you added one. (-> L12.1
    # Q7)
    "03.4": """
""",

    # ---- notebook 04 · Recovering the Inertia ------------------------------------
    # 04.1 In section 1 machine 1's inertia moved from a guess twice too large
    # towards the true value, while machine 0's barely moved. Explain why the
    # data says almost nothing about machine 0, and what that means for which
    # inertia a frequency measurement in DK2 can reveal: the local plant's or
    # the synchronous area's. (-> L12.1 Q7, Q9)
    "04.1": """
""",
    # 04.2 TODO 1 fitted a pre-fault window. Say what H came out as, how much
    # it depended on the initial guess, and whether the loss was any worse.
    # Then name the diagnostic you would report alongside H so that a reader
    # with no true value could tell the fit was meaningless. (-> L12.1 Q6)
    "04.2": """
""",
    # 04.3 The three RoCoF windows gave roughly -0.21, -0.19 and -0.14 Hz/s
    # against a true initial slope of -0.28 Hz/s. Explain the bias from the
    # shape of the curve, say which window you would quote, and say why a
    # quantity set by inertia is invisible to any power flow calculation. (->
    # L12.1 Q9)
    "04.3": """
""",
    # 04.4 WLS has run for fifty years. Across notebooks 01 to 04, name the
    # three things the physics residual gave you that WLS could not, and one
    # thing WLS or a Kalman filter still does better. Would you ask an operator
    # to replace WLS, or to run this beside it? (-> L12.1 Q10)
    "04.4": """
""",
}


In [ ]:
# notebook-questions v1 -- add the questions and your answers to the report ----------
# Safe to run again: it replaces the section rather than adding a second copy.
import os

NOTEBOOK_QUESTIONS = {
    "01.1": ('01', 'Weighted Least Squares, the Baseline', 'With the default set WLS converged with J around 9.5 and a worst |V| error of a few thousandths. With the thin set it converged just as readily and was roughly 45 times worse, worst where there were no meters. WLS has run for fifty years: say what these two runs show it does well, what it cannot do, and why its convergence report gave you no warning.', 'L12.1 Q10'),
    "01.2": ('01', 'Weighted Least Squares, the Baseline', 'The thin set gives rank 4 of the 10 unknowns. Say what observability depends on and what it does not, and use this set to explain why a more accurate meter at the same two buses would not have helped. What was the ridge term deciding at the unmetered buses?', 'L12.1 Q3'),
    "01.3": ('01', 'Weighted Least Squares, the Baseline', 'Section 1 could score every estimate only because it manufactured the readings from a known truth. In TODO 1 there was still a way to find the bad meter. Name the diagnostic, say how it checks an estimate without any ground truth, and name one error it cannot catch however large it is.', 'L12.1 Q6'),
    "01.4": ('01', 'Weighted Least Squares, the Baseline', "TODO 2 tried every one-meter addition to the thin set. Explain the winning placement in terms of the graph (which buses it ties to a measurement, which unobservable part it breaks) rather than the size of that bus's load, and say why an exhaustive search is the right claim to make on six buses and not on six hundred.", 'L12.1 Q3'),
    "02.1": ('02', 'Estimation With the Physics as a Residual', 'At $\\lambda_{\\mathrm{pf}} = 0$ this estimator is least squares on the meters; very large, it becomes a power flow solution that ignores them. Use the two ends of your $\\lambda$ sweep to explain why the power flow equations enter here as a residual to be satisfied rather than as a simulation you run, and why the trade-off only showed at the unmetered buses.', 'L12.1 Q1'),
    "02.2": ('02', 'Estimation With the Physics as a Residual', 'On the same thin set WLS left the unmetered buses about 3.0e-2 out. Explain how adding $\\mathcal{L}_{\\mathrm{pf}}$ gives the optimiser information at a bus with no meter at all, and why the result there must still be reported as an inference and not as a reading.', 'L12.1 Q4, Q3'),
    "02.3": ('02', 'Estimation With the Physics as a Residual', 'TODO 2 gave the estimator an admittance matrix with a branch missing. Say what the admittance matrix contains, where it comes from, and which part of this estimator, if any, is learned. Then use the wrong-Y run to explain why the loss did not warn you, and what a control room could check instead.', 'L12.1 Q2, Q6'),
    "02.4": ('02', 'Estimation With the Physics as a Residual', "Two things are built in here: the slack angle, and a sigmoid bound of 0.90 to 1.10 per unit on the voltage magnitudes. Say why the slack angle can be built in exactly, why the lecture's 0.94 to 1.06 operating band must only be checked, and what the wider bound would hide if a bus really fell below 0.90. Then say what a graph network would add over this direct solve, and why the notebook uses none.", 'L12.1 Q5, Q8'),
    "03.1": ('03', 'Letting the Grid Move', "Notebooks 01 and 02 treated the grid as a snapshot. Using section 1's trajectory and section 2's critical clearing time of about 0.33 s, say what the swing equation adds over power flow: name a question an operator needs answered that the power flow equations cannot answer at all.", 'L12.1 Q9'),
    "03.2": ('03', 'Letting the Grid Move', "Frequency is measured at scattered instants and the load angle is never measured anywhere. Explain how the swing residual turns the frequency record into an angle trajectory, and why neither the data term alone nor the residual alone would determine it. How is that the same move as notebook 02's unmetered buses?", 'L12.1 Q4, Q9'),
    "03.3": ('03', 'Letting the Grid Move', "There are no labels for $\\delta$, so TODO 2's residual between samples is your check. Where was it largest, and what would you conclude if it were large everywhere? Say how this and notebook 01's normalised residuals together let you check an estimate with no ground truth.", 'L12.1 Q6'),
    "03.4": ('03', 'Letting the Grid Move', "Machine 0 stands for the Nordic system seen through the Swedish link, with an inertia so large that it barely moves. Explain why eastern Denmark's frequency is set by that system and not by western Denmark's, and how an HVDC link would enter this model if you added one.", 'L12.1 Q7'),
    "04.1": ('04', 'Recovering the Inertia', "In section 1 machine 1's inertia moved from a guess twice too large towards the true value, while machine 0's barely moved. Explain why the data says almost nothing about machine 0, and what that means for which inertia a frequency measurement in DK2 can reveal: the local plant's or the synchronous area's.", 'L12.1 Q7, Q9'),
    "04.2": ('04', 'Recovering the Inertia', 'TODO 1 fitted a pre-fault window. Say what H came out as, how much it depended on the initial guess, and whether the loss was any worse. Then name the diagnostic you would report alongside H so that a reader with no true value could tell the fit was meaningless.', 'L12.1 Q6'),
    "04.3": ('04', 'Recovering the Inertia', 'The three RoCoF windows gave roughly -0.21, -0.19 and -0.14 Hz/s against a true initial slope of -0.28 Hz/s. Explain the bias from the shape of the curve, say which window you would quote, and say why a quantity set by inertia is invisible to any power flow calculation.', 'L12.1 Q9'),
    "04.4": ('04', 'Recovering the Inertia', 'WLS has run for fifty years. Across notebooks 01 to 04, name the three things the physics residual gave you that WLS could not, and one thing WLS or a Kalman filter still does better. Would you ask an operator to replace WLS, or to run this beside it?', 'L12.1 Q10'),
}

report_md = os.path.join(pb.RESULTS, "Ex12.1_report.md")
HEAD = "## Questions from the notebooks"
if not os.path.exists(report_md):
    print("No Ex12.1_report.md yet: run the cell that writes the report first.")
else:
    text = open(report_md, encoding="utf-8").read()
    text = text.split("\n" + HEAD)[0].rstrip() + "\n"
    out = ["", HEAD, "",
           "Each question is tagged with the lecture question it serves.", ""]
    missing, current = [], None
    for key, (nb, title, question, ref) in NOTEBOOK_QUESTIONS.items():
        if nb != current:
            out += [f"### Notebook {nb} · {title}", ""]
            current = nb
        answer = NOTEBOOK_ANSWERS.get(key, "").strip()
        if not answer:
            missing.append(key)
        out += [f"**{key}.** {question} *(→ {ref})*", "",
                answer or "*not answered*", ""]
    with open(report_md, "w", encoding="utf-8") as fh:
        fh.write(text + "\n".join(out))
    print(f"added to {report_md}: {len(NOTEBOOK_QUESTIONS) - len(missing)} of "
          f"{len(NOTEBOOK_QUESTIONS)} answered")
    if missing:
        print("not answered:", ", ".join(missing))


**What you should see.** `added to .../Ex12.1_report.md: 16 of 16 answered`.
Until then it lists the questions still empty, and the report says *not
answered* under each of them — to the marker too. Run this cell again after any
change to the report above it, because rewriting the report removes the
section; then make the PDF.